# Track 10 — Capstone: Customer Support Router (고객지원 티켓 라우팅 + HITL 발송)

## 티켓 라우팅 + HITL이란?

고객지원 자동화는 티켓을 올바른 큐로 보내고, 답변 발송처럼 되돌리기 어려운 행동을 안전하게 처리해야 합니다. 이 캡스톤은 규칙 기반 라우터와 HITL 발송 게이트를 한 흐름으로 묶습니다.

- **라우팅:** 티켓 본문을 `billing`·`it`·`general` 큐로 분류합니다.
- **초안 생성:** 큐별 고정 답변 초안을 만듭니다.
- **HITL 승인:** 사람이 실제 발송 본문을 보고 승인할 때만 `send_reply`가 성공합니다.
- **패키지:** 라우팅 정확도, HITL 결과, 정적 메트릭을 저장합니다.

## 이 노트북에서 보여줄 것

| Session | 보여주는 것 | 목적 |
|---|---|---|
| 1. Setup | facade 시작 · 골든 로더 · 정적 메트릭 헬퍼 · 패키지 저장 헬퍼 | 공통 준비 |
| 2. Router + HITL send | 라우팅 → 초안 → 사람 승인/거부 → 승인 시 발송 | 되돌리기 어려운 발송에 사람 판단 추가 |
| 3. 패키지 | 라우팅 정확도 + HITL 결과 + 정적 메트릭 저장 | 제출·회귀 산출물 마감 |

## 이 노트북을 마치면

- 티켓 라우팅과 발송 승인 게이트를 EXAONE 도구 레지스트리로 구성할 수 있습니다.
- 승인 신호를 클로저로 캡처해 모델이 도구 인자로 위조하지 못하게 막을 수 있습니다.
- 라우팅·HITL·정적 메트릭을 패키지로 묶어 회귀 기준선으로 남길 수 있습니다.

**산출물:** `_out/05/capstone_package.json`  
**실행 조건:** 라우팅·HITL·정적 메트릭은 모두 규칙/스텁 기반이라 API 키 없이 실행됩니다.

> 요약: 고객지원 티켓을 라우팅하고, 사람이 승인한 답변만 발송하도록 묶는 HITL 캡스톤입니다.


## Session 1. Setup


### Session 1-1. Setup

**하는 일:** 경로·클라이언트·공통 헬퍼를 준비합니다.

**정상:** `exaone 0.1.0 | HAS_API True` 한 줄이 출력됩니다(키가 없으면 `HAS_API False`). 이때 `ROOT`/`DATA`/`TRACK10` 경로 변수와 `client`, 그리고 골든 로더·정적 메트릭·패키지 저장 헬퍼가 메모리에 정의됩니다.

**의미:** 이후 단계에서 쓸 경로·클라이언트가 맞는지 먼저 봅니다.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import logging

# (en) Quiet library logs so the notebook output stays readable.
# (kr) 라이브러리 로그를 줄여 노트북 출력을 읽기 쉽게 한다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)

# (en) Facade-only startup; requires editable install at the repo root.
# (kr) `exaone` facade로 시작한다. 저장소 루트에서 editable 설치가 필요하다.
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "`exaone`이 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
ROOT = exaone.project_root()
TRACK10 = ROOT / "recipes" / "track10_ax_capstones"
DATA = TRACK10 / "data"
API_KEY = os.environ.get("EXAONE_API_KEY", "").strip()
BASE_URL = os.environ.get("EXAONE_BASE_URL", "").strip() or "http://localhost:8000/v1"
MODEL = (
    os.environ.get("EXAONE_MODEL", "").strip() or exaone.llm.ExaoneClient.DEFAULT_MODEL
)
HAS_API = bool(API_KEY)
client = None
if HAS_API:
    client = exaone.llm.ExaoneAPIClient(base_url=BASE_URL, model=MODEL, api_key=API_KEY)
print("exaone", exaone.__version__, "| HAS_API", HAS_API)


def load_capstone_golden(tag: str) -> list[dict]:
    # (en) Load golden rows for this capstone id or shared "all" rows.
    # (kr) 해당 캡스톤과 공통 "all" 골든 사례를 함께 불러온다.
    rows: list[dict] = []
    for line in (
        (DATA / "capstone_golden.jsonl").read_text(encoding="utf-8").splitlines()
    ):
        if not line.strip():
            continue
        row = json.loads(line)
        if row.get("capstone") in (tag, "all"):
            rows.append(row)
    return rows


def regression_m1_m6_m9(rows: list[dict]) -> dict:
    # (en) Static fixture metric demo (M1/M6/M9) over golden rows — metric MECHANICS, not live agent.
    # (kr) 정적 골든 fixture로 M1/M6/M9를 계산한다. 라이브 에이전트 성능이 아니라 메트릭 동작 예시다.
    from eval.metrics.m1_task_success import TaskGold
    from eval.metrics.m6_schema_adherence import SchemaSpec
    from eval.metrics.m9_faithfulness import LengthRatioJudge
    from eval.metrics import m1_task_success, m6_schema_adherence
    from eval.metrics.types import TrialResult

    m1s, m6s, m9s = [], [], []
    cases = []
    for row in rows:
        tid = row["id"]
        content = row.get("trial_content") or str(row.get("expected_answer", ""))
        tr = TrialResult(
            trial_id=f"cap-{tid}",
            task_id=tid,
            dataset="track10.golden",
            runner="capstone",
            final_content=content,
        )
        m1 = m6 = m9 = None
        if row.get("expected_answer") is not None:
            m1 = m1_task_success.score_trial_exact(
                tr, TaskGold(task_id=tid, answer=row["expected_answer"])
            )
            m1s.append(m1)
        rk = row.get("required_keys")
        if rk:
            _, loose = m6_schema_adherence.score_trial(tr, SchemaSpec(required_keys=rk))
            m6 = loose
            m6s.append(1.0 if loose else 0.0)
        if row.get("grounding_context"):
            m9 = LengthRatioJudge()(
                trial=tr, gold={"context": row["grounding_context"]}
            )
            m9s.append(m9)
        cases.append({"id": tid, "M1": m1, "M6": m6, "M9": m9})
    mean = lambda xs: sum(xs) / len(xs) if xs else 0.0
    return {
        "n": len(rows),
        "M1_mean": mean(m1s),
        "M6_loose_mean": mean(m6s),
        "M9_mean": mean(m9s),
        "cases": cases,
    }


def save_package(capstone_nb: str, body: dict) -> Path:
    # (en) Write capstone_package.json under <track>/_out/<nb>/ (absolute path, CWD-independent).
    # (kr) 절대경로로 <track>/_out/<nb>/capstone_package.json을 저장한다(커널 CWD와 무관).
    out_dir = TRACK10 / "_out" / capstone_nb
    out_dir.mkdir(parents=True, exist_ok=True)
    slo = exaone.observability.SLOSpec(
        name=f"capstone-{capstone_nb}",
        p95_chat_latency_ms=8000,
        structured_output_success_min="95%",
        notes="Track 10 capstone — adjust per deployment.",
    )
    payload = {
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "capstone_id": capstone_nb,
        "slo": slo.to_dict(),
        **body,
    }
    path = out_dir / "capstone_package.json"
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved", path.resolve())
    return path

**출력 해석:** `exaone 0.1.0 | HAS_API True` 한 줄만 찍히면 facade 시작과 헬퍼 정의가 모두 끝난 상태입니다.

- `HAS_API True`는 `EXAONE_API_KEY`가 있어 `client`가 만들어졌다는 뜻이지만, 이 캡스톤의 라우팅·HITL·정적 메트릭 경로는 규칙/스텁 기반이라 실제로는 LLM을 호출하지 않습니다 — 키 없이 `HAS_API False` 라도 끝까지 실행됩니다.
- 화면에는 버전 한 줄만 보이고 경로 변수(`ROOT`·`DATA`·`TRACK10`)와 헬퍼 함수는 출력 없이 정의만 되므로, 이 셀의 성공은 "다음 셀이 `DATA`와 헬퍼를 바로 쓸 수 있다"는 토대 확보를 뜻합니다.


## Session 2. Router + HITL send

**테스트 시나리오** — `support_inbox` (`support_inbox.jsonl`) 티켓 3건을 라우팅한 뒤, 큐별 답변 초안을 만들어 **사람이 그 내용을 보고 승인/거부**(mock `stdin`)한 결과로 `send_reply` 발송 여부가 갈리는 흐름을 봅니다.


### Session 2-1. Router + HITL send

**하는 일:** 라우팅 → 큐별 초안 → 사람이 내용 검토·승인 → 승인 시에만 발송, 한 흐름으로 실행합니다.

**정상 출력 (위 → 아래):**

1. `시나리오: 고객지원 티켓 3 건` + 미리보기 3줄 + 라우팅 리스트
2. `HITL 발송 게이트 — 사람 승인 소스: 기록된 키 재생(mock stdin)`
3. 티켓마다 3줄씩 — `[승인 요청] … 발송 본문: '…'` → `└ 사람 입력(stdin): 'y'/'n' → 승인/거부` → `→ t0x: 결과`
   - `t01`·`t02` → 입력 `'y'` → `sent:t01` / `sent:t02`
   - `t03` → 입력 `'n'` → `rejected_by_human`
4. `blocked extra-arg: Invalid arguments: Additional properties are not allowed ('approve' was unexpected)`
5. `assert` 3개 통과 → **무출력**

**의미:** 사람 입력(기본은 mock `stdin`, `INTERACTIVE=True` 면 실제 `input()`)이 발송 직전 실제 내용과 함께 개입하는 HITL 의 본질을 보여줍니다.


In [ ]:
from dataclasses import dataclass
from typing import Any, Callable

tickets = [
    json.loads(l)
    for l in (DATA / "support_inbox.jsonl").read_text(encoding="utf-8").splitlines()
    if l.strip()
]
print("시나리오: 고객지원 티켓", len(tickets), "건 (support_inbox.jsonl)")
for t in tickets[:3]:
    print(f"  {t.get('id', '?')}:", (t.get("text") or "")[:50])


def route_ticket(text: str) -> str:
    # (en) Keyword router: map ticket text to a queue (billing / it / general; else → default general).
    # (kr) 키워드 라우터: 티켓 본문을 큐로 매핑한다(billing / it / general; 미매칭은 기본값 general).
    t = text.lower()
    if "환불" in t or "결제" in t:
        return "billing"
    if "vpn" in t or "접속" in t:
        return "it"
    return "general"


# (en) Queue-specific reply drafts (deterministic templates; no LLM) — the concrete content a human reviews.
# (kr) 큐별 답변 초안(결정론 템플릿; LLM 불요) — 사람이 검토할 실제 내용.
REPLY_DRAFTS = {
    "billing": "결제/환불 문의 확인 중입니다. 영업일 기준 1~2일 내 처리 결과를 안내드리겠습니다.",
    "it": "IT 지원팀입니다. VPN 접속 오류 메시지와 발생 시각을 알려주시면 바로 확인하겠습니다.",
    "general": "문의해 주셔서 감사합니다. 담당 부서로 연결해 확인 후 회신드리겠습니다.",
}

routed = [
    {"id": t["id"], "route": route_ticket(t["text"]), "expected": t["expected_route"]}
    for t in tickets
]
print(routed)


@dataclass
class HumanApprovalController:
    # (en) HITL gate: shows the proposed reply, reads the operator's y/N keystroke via `stdin`, echoes it.
    # (kr) HITL 게이트: 제안 답변을 보여주고 운영자의 y/N 키 입력을 `stdin` 으로 읽어 그대로 에코한다.
    stdin: Callable[[], str]

    def request_approval(self, tool_name: str, args: dict) -> bool:
        print(f"  [승인 요청] {args['ticket_id']} 발송 본문: {args['body']!r}")
        answer = (self.stdin() or "").strip()
        approved = answer.lower() in ("y", "yes")
        print(f"  └ 사람 입력(stdin): {answer!r} → {'승인' if approved else '거부'}")
        return approved


def make_send_reply(
    controller: HumanApprovalController,
) -> Callable[[str, dict[str, Any]], dict[str, Any]]:
    # (en) Closure captures the controller, so send args stay schema-clean (no hidden approval flags).
    # (kr) 컨트롤러를 클로저로 캡처해, 발송 인자는 스키마에 맞는 값만 유지한다(숨은 승인 플래그 없음).
    def _exec(_n: str, args: dict) -> dict:
        if not controller.request_approval("send_reply", args):
            return exaone.tools.ToolResult.failure(
                source="send_reply", error="rejected_by_human"
            ).to_dict()
        return exaone.tools.ToolResult.success(
            content=f"sent:{args['ticket_id']}", source="send_reply"
        ).to_dict()

    return _exec


SEND_SCHEMA = {
    "type": "function",
    "function": {
        "name": "send_reply",
        "description": "Send customer reply (HITL)",
        "parameters": {
            "type": "object",
            "required": ["ticket_id", "body"],
            "properties": {"ticket_id": {"type": "string"}, "body": {"type": "string"}},
            "additionalProperties": False,
        },
    },
}


def send_reply_registry(
    controller: HumanApprovalController,
) -> "exaone.tools.ToolRegistry":
    # (en) Build a one-tool registry bound to this approval controller.
    # (kr) 이 승인 컨트롤러에 연결된 단일 도구 레지스트리를 만든다.
    reg = exaone.tools.ToolRegistry()
    reg.register(
        exaone.tools.tool_from_callable(
            "send_reply", SEND_SCHEMA, make_send_reply(controller)
        )
    )
    return reg


# (en) WHERE the human decision comes from. INTERACTIVE=True → actually type y/N via input() at runtime;
# (en) default (False) replays a recorded keystroke per ticket so the run stays deterministic & key-free.
# (kr) 사람 결정이 '어디서' 오는가. INTERACTIVE=True → 실행 중 input() 으로 직접 y/N 타이핑;
# (kr) 기본값(False)은 티켓별 기록된 키 입력을 재생해 결정론·키 불필요로 돌린다.
INTERACTIVE = False
RECORDED_DECISIONS = {"t01": "y", "t02": "y", "t03": "n"}


def decision_source(ticket_id: str) -> Callable[[], str]:
    # (en) Production wiring is literally `input(...)`; the notebook default replays a recorded key.
    # (kr) 운영 배선은 말 그대로 `input(...)`; 노트북 기본은 기록된 키를 재생한다.
    if INTERACTIVE:
        return lambda: input("  승인하시겠습니까? [y/N]: ")
    return lambda tid=ticket_id: RECORDED_DECISIONS.get(tid, "n")


# (en) End-to-end per ticket: route → draft → human reviews the content & approves → send only if approved.
# (kr) 티켓별 엔드투엔드: 라우팅 → 초안 → 사람이 내용 검토·승인 → 승인 시에만 발송.
print(
    "HITL 발송 게이트 — 사람 승인 소스:",
    "input() 실시간 입력" if INTERACTIVE else "기록된 키 재생(mock stdin)",
)
hitl_log = []
for r in routed:
    tid, route = r["id"], r["route"]
    draft = REPLY_DRAFTS[route]
    out = send_reply_registry(
        HumanApprovalController(stdin=decision_source(tid))
    ).execute("send_reply", {"ticket_id": tid, "body": draft})
    print(f"  → {tid}: {out['content'] or out['error']}")
    hitl_log.append(
        {
            "id": tid,
            "route": route,
            "approved": out["ok"],
            "result": out["content"] or out["error"],
        }
    )

# (en) Security check: an approval flag injected into the tool args is schema-rejected before _exec runs.
# (kr) 보안 점검: 도구 인자에 승인 플래그를 끼워 넣으면 _exec 실행 전에 스키마가 거부한다.
blocked = send_reply_registry(HumanApprovalController(stdin=lambda: "y")).execute(
    "send_reply", {"ticket_id": "t01", "body": REPLY_DRAFTS["billing"], "approve": True}
)
print("blocked extra-arg:", blocked["error"])

hitl_ok = {"per_ticket": hitl_log, "schema_block": blocked["error"]}
# (en) Regression guard: t01/t02 approved→sent, t03 rejected→rejected_by_human, injected flag schema-rejected.
# (kr) 회귀 가드: t01/t02 승인→sent, t03 거부→rejected_by_human, 끼워넣은 플래그는 스키마 거부.
assert [h["approved"] for h in hitl_log] == [True, True, False]
assert (
    hitl_log[0]["result"] == "sent:t01" and hitl_log[2]["result"] == "rejected_by_human"
)
assert "Additional properties" in blocked["error"]

**출력 해석:** 라우팅 → 초안 → 사람 승인 → 발송 여부가 한 흐름으로 보입니다.

- **라우팅:** `t01→billing`, `t02→it`, `t03→general`로 모두 기대값과 일치합니다.
- **승인 요청:** `[승인 요청] … 발송 본문`이 사람이 검토할 실제 내용을 보여줍니다.
- **사람 입력:** `stdin` 값이 그대로 출력됩니다. `t01`·`t02`는 `'y'`라 `sent`, `t03`은 `'n'`이라 `rejected_by_human`입니다.
- **입력 소스 교체:** 기본은 mock `stdin`입니다. `INTERACTIVE=True`이면 같은 게이트가 실제 `input()`을 읽습니다.
- **거부 판정:** `outcome`이 아니라 `error == "rejected_by_human"`을 봅니다.
- **승인 위조 차단:** 도구 인자에 `approve`를 넣으면 스키마가 거부합니다. 승인은 클로저 컨트롤러에서만 옵니다.


## Session 3. 패키지


### Session 3-1. 패키지

**하는 일:** 회귀 점수를 출력하고, 회귀·제출용 패키지 JSON을 저장합니다.

**정상:** `regression n=22 | M1=0.57 M6=0.50 M9(stub)=0.53`과 `saved …/_out/05/capstone_package.json` 두 줄이 출력됩니다. 저장된 JSON 에는 라우팅 결과·정확도·`hitl_send` 결과·M1/M6/M9 정적 메트릭·`SLOSpec`이 함께 묶입니다.

**의미:** 캡스톤 제출·회귀용 결과를 화면으로 확인하고 한 파일로 묶습니다.

In [ ]:
regression = regression_m1_m6_m9(load_capstone_golden("05"))
# (en) Surface the static metric-demo numbers inline (metric MECHANICS, not the router's score).
# (kr) 정적 메트릭 데모 수치를 화면에 보여준다(라우터 성능이 아니라 메트릭 동작 예시).
print(
    f"regression n={regression['n']} | M1={regression['M1_mean']:.2f} M6={regression['M6_loose_mean']:.2f} M9(stub)={regression['M9_mean']:.2f}"
)
accuracy = sum(1 for r in routed if r["route"] == r["expected"]) / len(routed)
package_path = save_package(
    "05",
    {
        "routing": routed,
        "routing_accuracy": accuracy,
        "hitl_send": hitl_ok,
        "regression": regression,
        "session_trace": [{"event": "route", "n": len(routed)}],
    },
)

**출력 해석:** `regression n=22 | M1=0.57 M6=0.50 M9(stub)=0.53`과 `saved …/_out/05/capstone_package.json` 두 줄이 출력되면 캡스톤 산출물이 절대경로에 기록된 것입니다.

- `regression` 수치는 공통 골든 `all` 22행을 채점하는 **메트릭 동작 데모**일 뿐 라우터 성능이 아닙니다(`05` 전용 행은 아직 없음): `M1=0.57` = `expected_answer`가 있는 7행 중 4건 정확 일치, `M6=0.50` = `required_keys`가 있는 6행 중 3건 키 충족, `M9(stub)=0.53` = `grounding_context`가 있는 5행에 대한 `LengthRatioJudge` 길이비 근사(충실도 아님, 테스트 전용 스텁). 라우터의 실제 신호는 `routing_accuracy`(이번 실행 1.0)·`hitl_send` 입니다.
- 저장된 패키지는 라우팅 결과·`routing_accuracy`·`hitl_send`(티켓별 `per_ticket` 승인/발송 결과 + 인자 위조 차단 `schema_block`)·`regression`·`SLOSpec`을 한 파일로 묶어 회귀 기준선으로 남깁니다.
- 경로는 `TRACK10/_out/05` 절대경로로 찍히므로 커널 CWD와 무관하게 항상 같은 위치에 저장됩니다(`_out/`은 gitignore 대상).

## 마무리

이 캡스톤에서는 고객지원 티켓을 규칙으로 라우팅하고, 큐별 답변 초안을 사람이 승인할 때만 발송하도록 구성한 뒤 `_out/05/capstone_package.json`으로 저장했습니다.

**핵심 정리**
- **라우팅:** `환불`/`결제`는 billing, `vpn`/`접속`은 it, 그 외는 general로 보냅니다.
- **HITL 발송:** 사람이 발송 본문을 보고 승인하면 `sent`, 거부하면 `rejected_by_human`입니다.
- **승인 위조 차단:** `approve` 같은 추가 인자는 스키마가 거부합니다. 승인 신호는 클로저 컨트롤러에서만 옵니다.
- **패키지:** 라우팅 결과, `routing_accuracy`, `hitl_send`, 정적 메트릭, SLO를 한 파일로 묶습니다.

**한계**
- 라우팅은 키워드 규칙이라 동의어·오타·복합 의도에 약합니다.
- 사람 승인은 기본적으로 mock `stdin`입니다. `INTERACTIVE=True`로 바꾸면 실제 `input()`을 사용할 수 있습니다.
- 답변 초안은 고정 템플릿입니다. 운영에서는 EXAONE 생성 답변을 사람이 검토하게 됩니다.
- 정적 메트릭은 fixture 채점 데모이며 라이브 에이전트 성능이 아닙니다.

**다음:** `10_06` Personal Agent. 운영화는 `10_07` 프로덕션 하네스를 참고하세요.

## 체크포인트

- [ ] Session 2 라우팅 → 초안 → 사람 승인 → 발송 확인
- [ ] Session 2 추가키 `approve` 스키마 거부 확인
- [ ] Session 3 `_out/05/capstone_package.json` 저장
